# Task 2.1: Feature Engineering on Rainfall Data

## 2.1.1 Spark Session

In [1]:
import socket
from functools import reduce

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

from pyspark.sql import SparkSession, Window, DataFrame

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pyspark.sql.functions as F
import seaborn as sns
import pandas as pd

import os
import sys

import time
from pyspark import StorageLevel
from pyspark.ml.functions import vector_to_array

In [ ]:
spark = SparkSession.builder.getOrCreate()

spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")
spark.conf.set("spark.sql.shuffle.partitions", "128")
spark.conf.set("spark.sql.adaptive.enabled", "true")

print("Running on:   ", socket.gethostname())
print("Spark master: ", spark.sparkContext.master)
print("Spark version:", spark.version)

Running on:    rainfall-dev-m
Spark master:  yarn
Spark version: 3.5.3


In [3]:
# Rainfall Multiplier per Day there are 12 five-minute intervals per hour and 24 hours per day

RF_MULTIPLIER = (24 * 60) / 5

## 2.1.2 Reading the Cleaned Rainfall Data

In [4]:
rain = spark.read.parquet("gs://dsa5208-rainfall-sixing-data/parquet/")

In [5]:
# Step 1: Extract the year and month from the observation date
rain = (
    rain.withColumn("date", F.to_date("date"))
        .withColumn("year", F.year("date"))
        .withColumn("month", F.month("date"))
)

# Step 2: Calculate mean rainfall per observed five-minute interval
# for each year and month
monthly_spk = (
    rain.filter(F.col("year").between(2017, 2024))
        .groupBy("year", "month")
        .agg(
            F.avg("reading_value").alias("mean_per_interval"),
            F.countDistinct("station_id").alias("n_stations")
        )
        .withColumn(
            "year_month",
            F.make_date("year", "month", F.lit(1))
        )
        .withColumn(
            "days",
            F.dayofmonth(F.last_day("year_month"))
        )
        # mean interval rainfall × expected intervals in the month
        .withColumn(
            "avg_rainfall_monthly",
            F.col("mean_per_interval") * F.col("days") * RF_MULTIPLIER
        )
        .orderBy("year_month")
)

## 2.1.3 Auto-Correlation of Rainfall and Its Timelags 

In [6]:
def lookback(seconds):
    return (Window.partitionBy("station_id")
            .orderBy("ts_long")
            .rangeBetween(-seconds, 0))

In [7]:
rain_lag = rain.withColumn("ts_long", F.col("timestamp").cast("long"))

w = Window.partitionBy("station_id").orderBy("ts_long")

lags = {"lag_5m": 1, "lag_30m": 6, "lag_1h": 12}

for name, k in lags.items():
    rain_lag = (rain_lag
        .withColumn(name, F.lag("reading_value", k).over(w))
        .withColumn(name + "_ts", F.lag("ts_long", k).over(w)))
    
    # null out where the gap isn't the expected duration
    rain_lag = rain_lag.withColumn(name,
        F.when(F.col("ts_long") - F.col(name + "_ts") == k * 300,
               F.col(name)))
    rain_lag = rain_lag.drop(name + "_ts")

In [8]:
past_30m = lookback(1500)     # Six readings: t−25 through t.
past_1h = lookback(3300)      # Twelve readings: t−55 through t.
history_1h = lookback(3600)   # Thirteen readings, including t−60 for lag_1h.

future_window = (
    Window.partitionBy("station_id")
          .orderBy("ts_long")
          .rangeBetween(1, 1800)
)

# Flag gaps that are not exactly five minutes.
rain_lag = (
    rain_lag
    .withColumn("_previous_ts", F.lag("ts_long").over(w))
    .withColumn(
        "_bad_gap",
        F.when(
            F.col("ts_long") - F.col("_previous_ts") == 300,
            0
        ).otherwise(1)
    )
)

rain_lag = (
    rain_lag

    # Historical rainfall features.
    .withColumn(
        "rain_cumsum_30m",
        F.sum("reading_value").over(past_30m)
    )
    .withColumn(
        "rain_cumsum_1h",
        F.sum("reading_value").over(past_1h)
    )
    .withColumn(
        "wet_count_30m",
        F.sum(
            (F.col("reading_value") > 0).cast("long")
        ).over(past_30m)
    )

    # Check the thirteen historical readings and twelve connecting gaps.
    .withColumn(
        "_history_count",
        F.count("reading_value").over(history_1h)
    )
    .withColumn(
        "_history_bad_gap",
        F.max("_bad_gap").over(past_1h)
    )

    # Check the six future readings and their connecting gaps.
    .withColumn(
        "_future_count",
        F.count("reading_value").over(future_window)
    )
    .withColumn(
        "_future_bad_gap",
        F.max("_bad_gap").over(future_window)
    )
    .withColumn(
        "future_30m",
        F.sum("reading_value").over(future_window)
    )

    # Use the observation timestamp consistently.
    .withColumn("hour", F.hour("timestamp"))
    .withColumn("month", F.month("timestamp"))
    .withColumn("year", F.year("timestamp"))
)

In [9]:
feature_cols = [
    "location_latitude",
    "location_longitude",
    "hour",
    "month",
    "reading_value",
    "lag_5m",
    "lag_30m",
    "lag_1h",
    "rain_cumsum_30m",
    "rain_cumsum_1h",
    "wet_count_30m",
    "thr"
]

THRESHOLDS = [0.0, 1.0, 2.5, 5.0, 10.0]
past_feature_cols = [c for c in feature_cols if c != "thr"]

base = (
    rain_lag
    .filter(
        (F.col("_history_count") == 13) &
        (F.col("_history_bad_gap") == 0) &
        (F.col("_future_count") == 6) &
        (F.col("_future_bad_gap") == 0)
    )
    .filter(F.col("year").between(2017, 2024))
    .select(
        "station_id", "timestamp", "year",
        *past_feature_cols,
        "future_30m"
    )
    .dropna(subset=past_feature_cols + ["future_30m"])

    # Store on executor disks instead of trying to hold everything in memory.
    .persist(StorageLevel.DISK_ONLY)
)

# Complete feature preparation now, so it is not included in model-fit timing.
# This first full-data preparation step can still take time.
started = time.perf_counter()

print("Eligible station-time rows:", base.count())
print(
    "Feature preparation seconds:",
    round(time.perf_counter() - started, 1)
)

target_end = F.col("timestamp") + F.expr("INTERVAL 30 MINUTES")

train = base.filter(
    (F.col("year") <= 2022) &
    (target_end < F.lit("2023-01-01").cast("timestamp"))
)

val = base.filter(
    (F.col("year") == 2023) &
    (target_end < F.lit("2024-01-01").cast("timestamp"))
)

test = base.filter(
    (F.col("year") == 2024) &
    (target_end < F.lit("2025-01-01").cast("timestamp"))
)

26/10/15 13:45:50 WARN SimpleTableFunctionRegistry: The function read_files replaced a previously registered function.


Eligible station-time rows: 47000882
Feature preparation seconds: 77.1


In [11]:
# 1. Takes a small uniform sample to reduce training time.
# 2. Gives positive and negative examples the same selection chance.
# 3. Prints their counts separately for every rainfall threshold.

SEED = 42

# These are starting fractions for checking that training runs successfully.
# Increase them later if too few positive examples are retained.
TRAIN_FRACTION = 0.005       # Approximately 0.5% of training station-times.
VALIDATION_FRACTION = 0.05   # Approximately 5% of validation station-times.
TEST_FRACTION = 0.10         # Approximately 10% of validation station-times. 

ML_PARTITIONS = 32


def prepare_sample(df, fraction):

    # STEP 1: Randomly select station-time observations.
    # Selection does not depend on whether rainfall exceeds a threshold.
    #
    # Historical features and future rainfall were already calculated.
    # Do not sample raw readings before calculating those windows.
    sampled = df.sample(
        withReplacement=False,
        fraction=fraction,
        seed=SEED
    )

    # STEP 2: Ask all five threshold questions for each selected observation.
    #
    # Example: if future_30m = 3 mm:
    # thr = 1 mm -> label 1
    # thr = 5 mm -> label 0
    labelled = (
        sampled
        .withColumn(
            "thr",
            F.explode(
                F.array(*[F.lit(float(t)) for t in THRESHOLDS])
            )
        )
        .withColumn(
            "label",
            (F.col("future_30m") > F.col("thr")).cast("double")
        )
    )

    # STEP 3: Keep only model inputs and the label.
    # Cache this smaller dataset so repeated training actions can reuse it.
    return (
        labelled
        .select(*feature_cols, "label")
        .repartition(ML_PARTITIONS)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )


train_final = prepare_sample(train, TRAIN_FRACTION)
val_final = prepare_sample(val, VALIDATION_FRACTION)
test_final = prepare_sample(test, TEST_FRACTION)


# STEP 4: Inspect imbalance separately for each threshold.
# A higher rainfall threshold may have fewer positive examples.
for dataset_name, dataset in [
    ("Training", train_final),
    ("Validation", val_final),
    ("Test", test_final)
]:
    summary = (
        dataset
        .groupBy("thr")
        .agg(
            F.count("*").alias("total"),
            F.sum("label").alias("positives")
        )
        .orderBy("thr")
        .collect()
    )

    # This action also fills the cache before model training starts.
    print(f"\n{dataset_name} class distribution")

    if not summary:
        raise ValueError(f"{dataset_name} sample is empty.")

    for row in summary:
        total = row["total"]
        positives = int(row["positives"])
        negatives = total - positives
        positive_pct = 100 * positives / total

        print(
            f"> {row['thr']:g} mm: "
            f"label 1 = {positives:,}, "
            f"label 0 = {negatives:,}, "
            f"positive percentage = {positive_pct:.2f}%"
        )

        # Stop if a threshold has no examples of one class.
        # Increase the relevant sample fraction and rerun this cell.
        if positives == 0 or negatives == 0:
            raise ValueError(
                f"{dataset_name}, threshold {row['thr']:g} mm: "
                "both classes are required. Increase the sample fraction."
            )

# IMPORTANT:
# Having both classes is only a minimum check.
# A handful of positive examples is not enough for reliable modelling.
#
# Keep validation and final test data at their natural class proportions.
# Do not balance them by selectively removing negative examples.

26/10/15 13:49:59 WARN CacheManager: Asked to cache already cached data.
26/10/15 13:49:59 WARN CacheManager: Asked to cache already cached data.
26/10/15 13:49:59 WARN CacheManager: Asked to cache already cached data.



Training class distribution
> 0 mm: label 1 = 10,283, label 0 = 159,366, positive percentage = 6.06%
> 1 mm: label 1 = 4,083, label 0 = 165,566, positive percentage = 2.41%
> 2.5 mm: label 1 = 2,403, label 0 = 167,246, positive percentage = 1.42%
> 5 mm: label 1 = 1,459, label 0 = 168,190, positive percentage = 0.86%
> 10 mm: label 1 = 690, label 0 = 168,959, positive percentage = 0.41%

Validation class distribution
> 0 mm: label 1 = 22,306, label 0 = 311,287, positive percentage = 6.69%
> 1 mm: label 1 = 8,860, label 0 = 324,733, positive percentage = 2.66%
> 2.5 mm: label 1 = 5,148, label 0 = 328,445, positive percentage = 1.54%
> 5 mm: label 1 = 3,013, label 0 = 330,580, positive percentage = 0.90%
> 10 mm: label 1 = 1,390, label 0 = 332,203, positive percentage = 0.42%



Test class distribution
> 0 mm: label 1 = 40,305, label 0 = 593,159, positive percentage = 6.36%
> 1 mm: label 1 = 16,260, label 0 = 617,204, positive percentage = 2.57%
> 2.5 mm: label 1 = 9,332, label 0 = 624,132, positive percentage = 1.47%
> 5 mm: label 1 = 5,310, label 0 = 628,154, positive percentage = 0.84%
> 10 mm: label 1 = 2,418, label 0 = 631,046, positive percentage = 0.38%


## 2.1.4 Output the Datasets to Parquet

In [ ]:
#train_final.write.mode("overwrite").parquet("gs://dsa5208-rainfall-sixing-data/train_parquet/")
#val_final.write.mode("overwrite").parquet("gs://dsa5208-rainfall-sixing-data/val_parquet/")
#test_final.write.mode("overwrite").parquet("gs://dsa5208-rainfall-sixing-data/test_parquet/")